<a href="https://colab.research.google.com/github/miguelterra98/Programacao_I/blob/main/aula_4_PROGRAMACAO_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import boto3
import rasterio
import matplotlib.pyplot as plt

In [ ]:
# Instala a biblioteca pystac_client se ainda não estiver instalada
!pip install pystac_client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 7.1 MB/s eta 0:00:00


In [ ]:
import pystac_client
import rasterio
import os

# 1. Define o URL do Catálogo STAC (Microsoft Planetary Computer é um bom exemplo para Sentinel-2)
STAC_CATALOG_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"

# 2. Define o ID da Coleção e o ID do Item a partir da sua solicitação
# ATENÇÃO: O item ID fornecido (2025) pode não existir, pois é uma data futura.
# Se o código falhar, considere usar um item de data passada para testes.
collection_id = "sentinel-2-l2a"
item_id = "S2B_MSIL2A_20250228T173149_R055_T14SLH_20250228T212633"

# 3. Cria um cliente STAC
print(f"Conectando ao catálogo STAC em: {STAC_CATALOG_URL}")
client = pystac_client.Client.open(STAC_CATALOG_URL)

# 4. Busca o Item STAC específico
print(f"Buscando item STAC: {item_id} na coleção {collection_id}...")
try:
    item = client.get_collection(collection_id).get_item(item_id, recursive=True)
    print(f"Item STAC encontrado: {item.id}")
except Exception as e:
    print(f"Erro ao buscar item STAC: {e}")
    print("O item pode não existir, especialmente se a data for futura (2025). ")
    print("Considere usar um item de data passada para testes, por exemplo: 'S2A_MSIL2A_20230303T131341_R095_T21HWE_20230303T200236'")
    raise # Re-lança a exceção para interromper a execução e mostrar o erro.

# 5. Identifica o asset relevante para download (ex: 'B02' para a banda Azul)
# Você pode listar todos os assets disponíveis com: print(item.assets.keys())
asset_key = 'B02' # Usando B02 como exemplo, como foi feito anteriormente.

if asset_key not in item.assets:
    print(f"Asset '{asset_key}' não encontrado no item. Assets disponíveis: {list(item.assets.keys())}")
    raise ValueError(f"Asset '{asset_key}' not found.")

asset = item.assets[asset_key]
image_url = asset.href

# 6. Define o nome do arquivo de saída e cria o diretório se não existir
output_dir = 'downloaded_stac_images'
os.makedirs(output_dir, exist_ok=True)
output_filename = f"{item.id}_{asset_key}.tif"
output_filepath = os.path.join(output_dir, output_filename)

# 7. Baixa a imagem usando rasterio
print(f"Baixando imagem '{asset_key}' de: {image_url}")
print(f"Salvando em: {output_filepath}")

try:
    # Usa rasterio para abrir o URL remoto e salvar localmente
    with rasterio.open(image_url) as src:
        # Copia o perfil (metadados) da imagem original para o novo arquivo
        profile = src.profile
        with rasterio.open(output_filepath, 'w', **profile) as dst:
            dst.write(src.read())
    print(f"Download de '{output_filename}' concluído com sucesso na pasta '{output_dir}'!")
except Exception as e:
    print(f"Erro ao baixar ou salvar a imagem: {e}")
    print("Verifique se o URL do asset é acessível e se há permissões de escrita.")


Conectando ao catálogo STAC em: https://planetarycomputer.microsoft.com/api/stac/v1
Buscando item STAC: S2B_MSIL2A_20250228T173149_R055_T14SLH_20250228T212633 na coleção sentinel-2-l2a...


/usr/local/lib/python3.12/dist-packages/pystac_client/collection_client.py:200: FallbackToPystac: Falling back to pystac. This might be slow.
  warnings.warn(FallbackToPystac())
